# Metric 3 — BGP Topology (T1): IPv6 AS-Level Topology Analysis

**TDDE35 Group 1 — David**

Replicates and extends the T1 metric from Czyz et al. (2015) using RouteViews BGP snapshots.

**What this notebook measures:**
- Number of ASes advertising IPv6 prefixes over time (2004–2025)
- Number of unique IPv6 prefixes in the routing table
- Hurricane Electric (AS 6939) presence in IPv6 AS-paths
- Comparison of IPv4 vs IPv6 routing table growth

## Google Colab Setup

**Run this section first when opening in Colab.**  
It mounts your Google Drive so that downloaded files and parsed results are saved there — meaning you never re-download across sessions.

In [ ]:
import os

IN_COLAB = 'COLAB_JUPYTER_IP' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # All cache lives in your Drive — survives across sessions
    BASE_DIR   = '/content/drive/MyDrive/bgp_metric3'
    CACHE_DIR  = BASE_DIR + '/bgp_cache'
    RIB_DIR    = BASE_DIR + '/rib_files'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(RIB_DIR,   exist_ok=True)
    print('Running in Colab — cache stored in Google Drive at', BASE_DIR)
else:
    # Local machine paths (unchanged)
    CACHE_DIR = './bgp_cache'
    RIB_DIR   = './rib_files'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(RIB_DIR,   exist_ok=True)
    print('Running locally — cache stored in', CACHE_DIR)

## 0. Install dependencies

In [2]:
# Install required packages
!pip install mrtparse requests matplotlib pandas tqdm


## 1. Imports and Configuration

In [ ]:
import mrtparse
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm.notebook import tqdm
import json
import os
import time

# ── Configuration ─────────────────────────────────────────────────────────────
COLLECTORS = {
    'route-views2': 'https://archive.routeviews.org/bgpdata/',
    'route-views6': 'https://archive.routeviews.org/route-views6/bgpdata/',
}

SAMPLE_YEARS = list(range(2004, 2026))
SAMPLE_MONTH = '01'
HE_ASN = 6939

# CACHE_DIR and RIB_DIR are set in the Colab Setup cell above.
# If running locally without that cell, set them here as fallback:
if 'CACHE_DIR' not in dir():
    CACHE_DIR = './bgp_cache'
    RIB_DIR   = './rib_files'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(RIB_DIR,   exist_ok=True)

print('Configuration loaded.')
print(f'Analysing years: {SAMPLE_YEARS[0]} – {SAMPLE_YEARS[-1]}')
print(f'Cache dir: {CACHE_DIR}')

## 2. Helper — Build RouteViews RIB URL

RouteViews stores RIB (Routing Information Base) dumps at predictable URLs:
```
https://archive.routeviews.org/bgpdata/YYYY.MM/RIBS/rib.YYYYMMDD.HHMM.bz2
```

In [4]:
def build_rib_url(year: int, month: str, day: str, collector: str = 'route-views2') -> str:
    """
    Build the URL for a RouteViews RIB snapshot.
    Tries 0000 hours first; RouteViews dumps at 0000, 0800, 1600 UTC.
    """
    base = COLLECTORS[collector]
    ym = f'{year}.{month}'
    date_str = f'{year}{month}{day}'
    return f'{base}{ym}/RIBS/rib.{date_str}.0000.bz2'


def url_exists(url: str) -> bool:
    """HEAD request to check if a file exists on RouteViews."""
    try:
        r = requests.head(url, timeout=10)
        return r.status_code == 200
    except Exception:
        return False


def find_rib_url(year: int, collector: str = 'route-views2') -> str | None:
    """
    Try a few days/times in January to find a valid RIB dump.
    Returns the first URL that exists, or None.
    """
    for day in ['02', '01', '03', '08']:
        for hour in ['0000', '0800', '1600']:
            base = COLLECTORS[collector]
            ym = f'{year}.{SAMPLE_MONTH}'
            date_str = f'{year}{SAMPLE_MONTH}{day}'
            url = f'{base}{ym}/RIBS/rib.{date_str}.{hour}.bz2'
            if url_exists(url):
                return url
    return None

# Quick test
test_url = find_rib_url(2023, 'route-views2')
print(f'Sample 2023 URL: {test_url}')

Sample 2023 URL: https://archive.routeviews.org/bgpdata/2023.01/RIBS/rib.20230102.0000.bz2


## 3. Parse a Single RIB Snapshot with bgpkit-parser

For each snapshot we extract:
- All **IPv6 prefixes** (prefix contains `:`)
- All **originating ASes** (last AS in the AS-path)
- All **AS-paths** containing HE (AS 6939)

In [ ]:
import urllib.request
import tempfile

def parse_rib_snapshot(url: str, year: int) -> dict:
    cache_file = os.path.join(CACHE_DIR, f'{year}_stats.json')
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)

    stats = {
        'ipv6_prefixes': set(),
        'ipv6_origin_ases': set(),
        'ipv6_total_paths': 0,
        'ipv6_he_paths': 0,
        'ipv4_prefixes': set(),
        'ipv4_origin_ases': set(),
    }

    try:
        print(f'  Downloading {year}...')
        tmp = tempfile.NamedTemporaryFile(suffix='.bz2', delete=False)
        urllib.request.urlretrieve(url, tmp.name)
        print(f'  Parsing {year}...')

        entry_count = 0
        for entry in mrtparse.Reader(tmp.name):
            entry_count += 1
            if entry_count % 50000 == 0:
                print(f'  {entry_count:,} entries processed...', end='\r')
            if entry.err:
                continue

            # subtype is a dict e.g. {2: 'RIB_IPV4_UNICAST'} or {4: 'RIB_IPV6_UNICAST'}
            subtype_val = list(entry.data.get('subtype', {0: ''}).keys())[0]

            prefix = entry.data.get('prefix', '')
            prefix_len = entry.data.get('prefix_length', 0)
            prefix_str = f'{prefix}/{prefix_len}'

            for rib in entry.data.get('rib_entries', []):
                as_path = []
                for attr in rib.get('path_attributes', []):
                    # type is also a dict e.g. {2: 'AS_PATH'}
                    type_val = list(attr.get('type', {0: ''}).keys())[0]
                    if type_val == 2:  # AS_PATH
                        for seg in attr.get('value', []):
                            as_path += seg.get('value', [])
                origin_as = as_path[-1] if as_path else None

                if subtype_val == 4:  # RIB_IPV6_UNICAST
                    stats['ipv6_prefixes'].add(prefix_str)
                    stats['ipv6_total_paths'] += 1
                    if origin_as:
                        stats['ipv6_origin_ases'].add(origin_as)
                    if str(HE_ASN) in as_path:
                        stats['ipv6_he_paths'] += 1
                elif subtype_val == 2:  # RIB_IPV4_UNICAST
                    stats['ipv4_prefixes'].add(prefix_str)
                    if origin_as:
                        stats['ipv4_origin_ases'].add(origin_as)

        os.unlink(tmp.name)

    except Exception as e:
        print(f'  [!] Error parsing {year}: {e}')
        return None

    result = {
        'year': year,
        'url': url,
        'ipv6_prefix_count':    len(stats['ipv6_prefixes']),
        'ipv6_origin_as_count': len(stats['ipv6_origin_ases']),
        'ipv6_total_paths':     stats['ipv6_total_paths'],
        'ipv6_he_paths':        stats['ipv6_he_paths'],
        'ipv6_he_fraction':     (
            stats['ipv6_he_paths'] / stats['ipv6_total_paths']
            if stats['ipv6_total_paths'] > 0 else 0
        ),
        'ipv4_prefix_count':    len(stats['ipv4_prefixes']),
        'ipv4_origin_as_count': len(stats['ipv4_origin_ases']),
    }

    with open(cache_file, 'w') as f:
        json.dump(result, f, indent=2)

    return result

print('parse_rib_snapshot() defined.')

parse_rib_snapshot() defined.


## 4. Run the Historical Collection (2004–2025)

⚠️ **This cell takes time** — each RIB file is ~1–4 GB compressed.
Results are cached in `./bgp_cache/` so re-runs are instant.

Tip: start with a subset (e.g. `SAMPLE_YEARS[-5:]`) to verify it works,
then run the full range overnight.

In [ ]:
import urllib.request

def download_rib(url: str, year: int) -> str:
    """Download RIB file if not already cached locally. Returns local path."""
    local_path = os.path.join(RIB_DIR, f'{year}.bz2')
    if os.path.exists(local_path):
        print(f'  Using cached file for {year}: {local_path}')
        return local_path
    print(f'  Downloading {year} from RouteViews...')
    urllib.request.urlretrieve(url, local_path)
    print(f'  Saved to {local_path}')
    return local_path

def parse_rib_snapshot(url: str, year: int) -> dict:
    """Parse a RIB snapshot. Uses JSON cache if available, local .bz2 if downloaded."""
    cache_file = os.path.join(CACHE_DIR, f'{year}_stats.json')
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)

    stats = {
        'ipv6_prefixes': set(),
        'ipv6_origin_ases': set(),
        'ipv6_total_paths': 0,
        'ipv6_he_paths': 0,
        'ipv4_prefixes': set(),
        'ipv4_origin_ases': set(),
    }

    try:
        local_path = download_rib(url, year)
        print(f'  Parsing {year}...')

        entry_count = 0
        for entry in mrtparse.Reader(local_path):
            entry_count += 1
            if entry_count % 50000 == 0:
                print(f'  {entry_count:,} entries processed...', end='\r')
            if entry.err:
                continue

            subtype_val = list(entry.data.get('subtype', {0: ''}).keys())[0]
            prefix = entry.data.get('prefix', '')
            prefix_len = entry.data.get('prefix_length', 0)
            prefix_str = f'{prefix}/{prefix_len}'

            for rib in entry.data.get('rib_entries', []):
                as_path = []
                for attr in rib.get('path_attributes', []):
                    type_val = list(attr.get('type', {0: ''}).keys())[0]
                    if type_val == 2:  # AS_PATH
                        for seg in attr.get('value', []):
                            as_path += seg.get('value', [])
                origin_as = as_path[-1] if as_path else None

                if subtype_val == 4:  # RIB_IPV6_UNICAST
                    stats['ipv6_prefixes'].add(prefix_str)
                    stats['ipv6_total_paths'] += 1
                    if origin_as:
                        stats['ipv6_origin_ases'].add(origin_as)
                    if HE_ASN in as_path:
                        stats['ipv6_he_paths'] += 1
                elif subtype_val == 2:  # RIB_IPV4_UNICAST
                    stats['ipv4_prefixes'].add(prefix_str)
                    if origin_as:
                        stats['ipv4_origin_ases'].add(origin_as)

    except Exception as e:
        print(f'  [!] Error parsing {year}: {e}')
        return None

    result = {
        'year': year,
        'url': url,
        'ipv6_prefix_count':    len(stats['ipv6_prefixes']),
        'ipv6_origin_as_count': len(stats['ipv6_origin_ases']),
        'ipv6_total_paths':     stats['ipv6_total_paths'],
        'ipv6_he_paths':        stats['ipv6_he_paths'],
        'ipv6_he_fraction':     (
            stats['ipv6_he_paths'] / stats['ipv6_total_paths']
            if stats['ipv6_total_paths'] > 0 else 0
        ),
        'ipv4_prefix_count':    len(stats['ipv4_prefixes']),
        'ipv4_origin_as_count': len(stats['ipv4_origin_ases']),
    }

    with open(cache_file, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Cached stats to {cache_file}')

    return result

print('parse_rib_snapshot() defined (with local .bz2 cache).')

In [ ]:
## Run full historical collection (2004–2025)
# JSON cache → instant; local .bz2 cache → skip download; otherwise downloads from RouteViews.
rows = []
for year in tqdm(SAMPLE_YEARS, desc='Years'):
    url = find_rib_url(year, 'route-views2')
    if url is None:
        print(f'[!] No RIB URL found for {year}, skipping.')
        continue
    row = parse_rib_snapshot(url, year)
    if row:
        rows.append(row)

df = pd.DataFrame(rows).sort_values('year').reset_index(drop=True)
print(f'\nDone. {len(df)} years collected.')
print(df[['year', 'ipv6_prefix_count', 'ipv4_prefix_count']].to_string(index=False))

## 5. Results Table

In [12]:
display_cols = [
    'year',
    'ipv6_prefix_count', 'ipv4_prefix_count',
    'ipv6_origin_as_count', 'ipv4_origin_as_count',
    'ipv6_he_fraction',
]

df_display = df[display_cols].copy()
df_display['ipv6_he_fraction'] = df_display['ipv6_he_fraction'].map('{:.1%}'.format)
df_display.columns = [
    'Year',
    'IPv6 Prefixes', 'IPv4 Prefixes',
    'IPv6 Origin ASes', 'IPv4 Origin ASes',
    'HE in IPv6 paths'
]
df_display

NameError: name 'df' is not defined

## 6. Plots

### 6a. IPv6 vs IPv4 Prefix Growth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Prefix counts ───────────────────────────────────────────────────────
ax = axes[0]
ax.plot(df['year'], df['ipv6_prefix_count'], 'o-', color='steelblue',
        label='IPv6 prefixes', linewidth=2, markersize=5)
ax2 = ax.twinx()
ax2.plot(df['year'], df['ipv4_prefix_count'], 's--', color='coral',
         label='IPv4 prefixes', linewidth=2, markersize=5)

ax.set_xlabel('Year')
ax.set_ylabel('IPv6 Prefix Count', color='steelblue')
ax2.set_ylabel('IPv4 Prefix Count', color='coral')
ax.set_title('IPv6 vs IPv4 Prefix Count (RouteViews)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax.grid(alpha=0.3)

# ── Right: Origin AS counts ───────────────────────────────────────────────────
ax = axes[1]
ax.plot(df['year'], df['ipv6_origin_as_count'], 'o-', color='steelblue',
        label='IPv6 origin ASes', linewidth=2, markersize=5)
ax3 = ax.twinx()
ax3.plot(df['year'], df['ipv4_origin_as_count'], 's--', color='coral',
         label='IPv4 origin ASes', linewidth=2, markersize=5)

ax.set_xlabel('Year')
ax.set_ylabel('IPv6 Origin AS Count', color='steelblue')
ax3.set_ylabel('IPv4 Origin AS Count', color='coral')
ax.set_title('IPv6 vs IPv4 Origin AS Count')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax3.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax3.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('metric3_prefix_as_growth.pdf', bbox_inches='tight')
plt.savefig('metric3_prefix_as_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved metric3_prefix_as_growth.pdf')

### 6b. Hurricane Electric (AS 6939) Dominance in IPv6 AS-Paths

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

he_pct = df['ipv6_he_fraction'] * 100

ax.fill_between(df['year'], he_pct, alpha=0.2, color='darkorange')
ax.plot(df['year'], he_pct, 'o-', color='darkorange', linewidth=2, markersize=6,
        label='AS 6939 (HE) in IPv6 paths')

# Annotate Czyz 2013 reference point (~95%)
ax.axhline(y=95, color='red', linestyle='--', linewidth=1, alpha=0.7,
           label='Czyz et al. 2013 baseline (~95%)')

ax.set_xlabel('Year')
ax.set_ylabel('% of IPv6 AS-Paths containing HE (AS 6939)')
ax.set_title('Hurricane Electric Dominance in IPv6 BGP Topology')
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('metric3_he_dominance.pdf', bbox_inches='tight')
plt.savefig('metric3_he_dominance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved metric3_he_dominance.pdf')

### 6c. IPv6 Share of Total Routing Table

In [ ]:
df['ipv6_share_prefixes'] = (
    df['ipv6_prefix_count'] /
    (df['ipv6_prefix_count'] + df['ipv4_prefix_count']) * 100
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(df['year'], df['ipv6_share_prefixes'], alpha=0.15, color='steelblue')
ax.plot(df['year'], df['ipv6_share_prefixes'], 'o-', color='steelblue',
        linewidth=2, markersize=6)

ax.set_xlabel('Year')
ax.set_ylabel('IPv6 share of total routing table (%)')
ax.set_title('IPv6 Share of BGP Routing Table (RouteViews)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('metric3_ipv6_share.pdf', bbox_inches='tight')
plt.savefig('metric3_ipv6_share.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved metric3_ipv6_share.pdf')

## 7. Save Results to CSV

In [ ]:
df.to_csv('metric3_bgp_results.csv', index=False)
print('Results saved to metric3_bgp_results.csv')
print(df[['year', 'ipv6_prefix_count', 'ipv6_origin_as_count',
          'ipv6_he_fraction', 'ipv4_prefix_count']].to_string(index=False))

## 8. Quick Sanity Check — Single Recent Year

Run this first to verify the pipeline works before the full 2004–2025 run.

In [6]:
# ── Sanity check: just parse 2023 ─────────────────────────────────────────────
test_year = 2023
url = find_rib_url(test_year, 'route-views2')
print(f'URL for {test_year}: {url}')

if url:
    row = parse_rib_snapshot(url, test_year)
    if row:
        print(f"\nIPv6 prefixes:      {row['ipv6_prefix_count']:,}")
        print(f"IPv6 origin ASes:   {row['ipv6_origin_as_count']:,}")
        print(f"HE in IPv6 paths:   {row['ipv6_he_fraction']:.1%}")
        print(f"IPv4 prefixes:      {row['ipv4_prefix_count']:,}")

URL for 2023: https://archive.routeviews.org/bgpdata/2023.01/RIBS/rib.20230102.0000.bz2
  Parsing 2023...
  950,000 entries processed...
IPv6 prefixes:      0
IPv6 origin ASes:   0
HE in IPv6 paths:   0.0%
IPv4 prefixes:      890,521


In [7]:
# Debug: peek at first few entries to see structure
import mrtparse, tempfile, urllib.request

url = 'https://archive.routeviews.org/bgpdata/2023.01/RIBS/rib.20230102.0000.bz2'
tmp = tempfile.NamedTemporaryFile(suffix='.bz2', delete=False)
urllib.request.urlretrieve(url, tmp.name)

count = 0
for entry in mrtparse.Reader(tmp.name):
    if entry.err:
        continue
    print(entry.data)
    count += 1
    if count >= 3:
        break

OrderedDict({'timestamp': {1672617600: '2023-01-02 01:00:00'}, 'type': {13: 'TABLE_DUMP_V2'}, 'subtype': {1: 'PEER_INDEX_TABLE'}, 'length': 1020, 'collector_bgp_id': '128.223.51.102', 'view_name_length': 11, 'view_name': 'VRF default', 'peer_count': 77, 'peer_entries': [OrderedDict({'peer_type': 2, 'peer_bgp_id': '0.0.0.0', 'peer_ip': '0.0.0.0', 'peer_as': '0'}), OrderedDict({'peer_type': 2, 'peer_bgp_id': '0.0.0.0', 'peer_ip': '4.68.4.46', 'peer_as': '3356'}), OrderedDict({'peer_type': 2, 'peer_bgp_id': '0.0.0.0', 'peer_ip': '5.101.110.2', 'peer_as': '14061'}), OrderedDict({'peer_type': 2, 'peer_bgp_id': '12.0.1.63', 'peer_ip': '12.0.1.63', 'peer_as': '7018'}), OrderedDict({'peer_type': 2, 'peer_bgp_id': '37.139.139.17', 'peer_ip': '37.139.139.17', 'peer_as': '57866'}), OrderedDict({'peer_type': 2, 'peer_bgp_id': '0.0.0.0', 'peer_ip': '43.226.4.1', 'peer_as': '63927'}), OrderedDict({'peer_type': 2, 'peer_bgp_id': '184.95.245.30', 'peer_ip': '45.61.0.85', 'peer_as': '22652'}), OrderedD